[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alanturin-g/Computational_Neuroscience_UNITO/blob/main/Perotti_4a_machine_learning.ipynb)

In [1]:
# Yes, again..
# File -> Save a copy in Drive
from google.colab import auth; auth.authenticate_user()
from google.colab import drive; drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install nibabel
!pip install nilearn
import tarfile
import os
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 70.6 MB/s eta 0:00:00


In [4]:
root = Path("/content/drive/MyDrive/neuroco_unzip/neurocon")

#Faccio un trimming dei primi 20.88 s(6 volumi)
# cartella dove salvare i file trimmati (deve esistere!)
outdir = Path("/content/drive/MyDrive/prova")         # <-- modifica con il tuo percorso

N = 6  # numero di volumi da rimuovere

print("Root:", root, "| exists:", root.exists())
print("Output:", outdir, "| exists:", outdir.exists())

files = list(root.rglob("*.nii*"))
print("NIfTI trovati:", len(files))




for f in files:
    print("\nProcessing:", f)

    try:
        img = nib.load(str(f))
        data = img.get_fdata(dtype=np.float32)

        # taglia i primi 6 volumi
        data_trim = data[..., N:]

        # salva con lo stesso nome
        out = outdir / f.name
        nib.save(nib.Nifti1Image(data_trim, img.affine, img.header), str(out))

        print("  → Salvato:", out)
    except EOFError:
        print(f"  → SKIPPED: {f.name} (Corrupted or incomplete file)")
    except Exception as e:
        print(f"  → ERROR processing {f.name}: {e}")

Root: /content/drive/MyDrive/neuroco_unzip/neurocon | exists: True
Output: /content/drive/MyDrive/prova | exists: True
NIfTI trovati: 74

Processing: /content/drive/MyDrive/neuroco_unzip/neurocon/sub-control032014/anat/sub-control032014_T1w.nii.gz
  → Salvato: /content/drive/MyDrive/prova/sub-control032014_T1w.nii.gz

Processing: /content/drive/MyDrive/neuroco_unzip/neurocon/sub-control032014/func/sub-control032014_task-resting_run-1_bold.nii.gz
  → Salvato: /content/drive/MyDrive/prova/sub-control032014_task-resting_run-1_bold.nii.gz

Processing: /content/drive/MyDrive/neuroco_unzip/neurocon/sub-control032014/func/sub-control032014_task-resting_run-2_bold.nii.gz
  → Salvato: /content/drive/MyDrive/prova/sub-control032014_task-resting_run-2_bold.nii.gz

Processing: /content/drive/MyDrive/neuroco_unzip/neurocon/sub-control032015/anat/sub-control032015_T1w.nii.gz
  → Salvato: /content/drive/MyDrive/prova/sub-control032015_T1w.nii.gz

Processing: /content/drive/MyDrive/neuroco_unzip/neuro

In [5]:

from nilearn.masking import compute_epi_mask
from nilearn.image import smooth_img

# === PERCORSI ===
root = Path("/content/drive/MyDrive/prova")         # cartella con i NIfTI già trimmati (4D)
outdir = Path("/content/drive/MyDrive/preprocessed")     # cartella di output "preprocessed"
outdir.mkdir(exist_ok=True, parents=True)
#Applico un filtro gaussiano 6 mm
FWHM_MM = 6.0

files = list(root.rglob("*bold.nii*"))
print("File trovati:", len(files))

for f in files:
    print("\nProcessing:", f)
    img = nib.load(str(f))
    data = img.get_fdata(dtype=np.float32)

    if data.ndim != 4:
        print("  → Skip (non 4D)")
        continue

    # 1) Maschera EPI dal volume medio
    mean_vol = data.mean(axis=3)
    mean_img = nib.Nifti1Image(mean_vol, img.affine, img.header)
    mask_img = compute_epi_mask(mean_img)  # 3D

    # 2) Applica maschera (fuori →

    mask = mask_img.get_fdata().astype(bool)
    data_masked = np.zeros_like(data, dtype=np.float32)
    data_masked[mask, :] = data[mask, :]

    masked_img = nib.Nifti1Image(data_masked, img.affine, img.header)

    # 3) Smoothing 6 mm FWHM (in mm)
    smoothed_img = smooth_img(masked_img, fwhm=FWHM_MM)


    # 4) Salva in "preproc" mantenendo nome + suffisso
    base = f.name.replace(".nii.gz", "").replace(".nii", "")
    out = outdir / f"{base}_mask_smooth6mm.nii.gz"
    nib.save(smoothed_img, str(out))
    print("  → Salvato:", out)



File trovati: 48

Processing: /content/drive/MyDrive/prova/sub-control032014_task-resting_run-1_bold.nii.gz
  → Salvato: /content/drive/MyDrive/preprocessed/sub-control032014_task-resting_run-1_bold_mask_smooth6mm.nii.gz

Processing: /content/drive/MyDrive/prova/sub-control032014_task-resting_run-2_bold.nii.gz
  → Salvato: /content/drive/MyDrive/preprocessed/sub-control032014_task-resting_run-2_bold_mask_smooth6mm.nii.gz

Processing: /content/drive/MyDrive/prova/sub-control032015_task-resting_run-1_bold.nii.gz
  → Salvato: /content/drive/MyDrive/preprocessed/sub-control032015_task-resting_run-1_bold_mask_smooth6mm.nii.gz

Processing: /content/drive/MyDrive/prova/sub-control032016_task-resting_run-1_bold.nii.gz
  → Salvato: /content/drive/MyDrive/preprocessed/sub-control032016_task-resting_run-1_bold_mask_smooth6mm.nii.gz

Processing: /content/drive/MyDrive/prova/sub-control032016_task-resting_run-2_bold.nii.gz
  → Salvato: /content/drive/MyDrive/preprocessed/sub-control032016_task-rest

In [6]:
!zip -r preprocessed.zip /content/drive/MyDrive/preprocessed

from google.colab import files
files.download("preprocessed.zip")


  adding: content/drive/MyDrive/preprocessed/ (stored 0%)
  adding: content/drive/MyDrive/preprocessed/.ipynb_checkpoints/ (stored 0%)
  adding: content/drive/MyDrive/preprocessed/sub-control032014_task-resting_run-1_bold_mask_smooth6mm.nii.gz (deflated 1%)
  adding: content/drive/MyDrive/preprocessed/sub-control032014_task-resting_run-2_bold_mask_smooth6mm.nii.gz (deflated 1%)
  adding: content/drive/MyDrive/preprocessed/sub-control032015_task-resting_run-1_bold_mask_smooth6mm.nii.gz (deflated 1%)
  adding: content/drive/MyDrive/preprocessed/sub-control032016_task-resting_run-1_bold_mask_smooth6mm.nii.gz (deflated 1%)
  adding: content/drive/MyDrive/preprocessed/sub-control032016_task-resting_run-2_bold_mask_smooth6mm.nii.gz (deflated 1%)
  adding: content/drive/MyDrive/preprocessed/sub-control032017_task-resting_run-1_bold_mask_smooth6mm.nii.gz (deflated 1%)
  adding: content/drive/MyDrive/preprocessed/sub-control032017_task-resting_run-2_bold_mask_smooth6mm.nii.gz (deflated 1%)
  ad

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>